<a href="https://colab.research.google.com/github/liqq1024/GenAI_LLMs_2026Fall/blob/main/colab-notebooks/Week10_RAG_and_LLM_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 10: RAG and LLM Evaluation

## Live Coding Demonstration (Google Colab)

In this lab, you will build a small document question-answering system and evaluate both retrieval quality and answer quality. The document collection includes direct-answer questions, multi-source questions, unsupported questions, and an archived policy that can create confusion.

### Learning goals

- Build a direct RAG pipeline: documents → chunks → embeddings → FAISS index → retrieval → grounded prompt → answer.
- Inspect evidence rather than trusting an answer automatically.
- Compare a base model, RAG with top-1 retrieval, and RAG with top-3 retrieval.
- Measure Hit@k and Precision@k. Hit@k: Did retrieval return at least one relevant document in the top \(k\)? It is binary: 0 or 1.
- Diagnose failures caused by documents, chunking, embeddings, retrieval, prompts, generation, or evaluation.

**Estimated lab time:** 45–60 minutes


## Section 1: Install Packages

In [1]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "-q", "install",
    "sentence-transformers", "faiss-cpu", "transformers", "sentencepiece"
])

import json
import textwrap
from collections import defaultdict

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline

np.random.seed(42)
print("Packages installed.")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


Packages installed.


## Section 2: Load Documents

These are fictional, simplified university documents created for this lab. The archived policy is intentionally included so you can examine how an outdated document can confuse retrieval or generation.

In [2]:
documents = [
    {
        "id": "policy_1",
        "title": "Late Assignment Policy (Current)",
        "status": "current",
        "text": """
Late Assignment Policy for the current academic year. Assignments may be submitted
up to six calendar days after the published deadline. Work submitted during the
first two calendar days late receives a ten percent deduction. Work submitted
during days three and four receives a twenty percent deduction. Work submitted
during days five and six receives a thirty percent deduction. Assignments are not
accepted after six calendar days unless a documented emergency has been approved
by the instructor before grading is completed. Students should submit incomplete
work by the deadline when possible because late penalties apply to the entire
submission. Technical problems should be reported promptly and do not automatically
extend the deadline. The instructor may publish a different schedule for a group
project only when that schedule is explicitly written in the assignment instructions.
""",
    },
    {
        "id": "policy_2",
        "title": "Exam Policy",
        "status": "current",
        "text": """
All examinations are closed book, closed notes, and closed internet unless the
instructor announces a specific exception in writing. Students may bring only
materials explicitly approved for that examination. Standard calculators are
permitted only when stated in the exam instructions. Students should arrive on
time because additional time is not provided for late arrival. A student who misses
an examination must contact the instructor as soon as possible and provide
documentation for an excused absence when requested. A make-up examination is not
automatic. If a make-up is approved, it may use a different format or different
questions while assessing the same learning objectives. Academic-integrity rules
apply to every examination and to all permitted materials.
""",
    },
    {
        "id": "policy_3",
        "title": "Attendance and Excused Absence Policy",
        "status": "current",
        "text": """
Students are expected to participate in scheduled class meetings and complete
announced activities. Students who have an illness, family emergency, religious
observance, university-sponsored activity, or other documented reason should notify
the instructor as early as possible. Documentation may be required for an excused
absence, particularly when the absence affects an exam, presentation, or deadline.
An excused absence does not automatically remove every course requirement. Instead,
the instructor and student will determine a reasonable alternative when feasible.
Students remain responsible for reviewing missed course material, monitoring course
announcements, and following the relevant assignment or examination policy. Requests
made long after an absence may be difficult to accommodate.
""",
    },
    {
        "id": "policy_4",
        "title": "Academic Integrity and Generative AI Policy",
        "status": "current",
        "text": """
Students must submit work that accurately represents their own learning and must
acknowledge permitted sources and tools. Generative AI may be used only when an
assignment explicitly permits it. When AI use is permitted, students must describe
the tool, the purpose of use, and the extent to which generated material was
incorporated. Students remain responsible for verifying facts, citations, code, and
reasoning. Copying generated output without understanding or acknowledgment may
violate academic-integrity expectations. For collaborative work, students must also
identify their own contributions. Questions about acceptable use should be directed
to the instructor before submission rather than after a concern is raised.
""",
    },
    {
        "id": "policy_archive",
        "title": "Late Assignment Policy (Archived 2024)",
        "status": "archived",
        "text": """
Archived policy for the 2024 academic year. At that time, assignments could be
submitted no more than three calendar days after the deadline. The late penalty was
fifteen percent per day. This document is retained for historical reference only
and does not govern current courses. Students and assistants should use the current
Late Assignment Policy for active courses. If an older document is retrieved,
the system should identify its archived status rather than treating it as current.
""",
    },
]

documents_df = pd.DataFrame([
    {"id": doc["id"], "title": doc["title"], "status": doc["status"],
     "word_count": len(doc["text"].split())}
    for doc in documents
])
documents_df

,id,title,status,word_count
0,policy_1,Late Assignment Policy (Current),current,134
1,policy_2,Exam Policy,current,114
2,policy_3,Attendance and Excused Absence Policy,current,106
3,policy_4,Academic Integrity and Generative AI Policy,current,102
4,policy_archive,Late Assignment Policy (Archived 2024),archived,77


## Section 3: Chunk Documents

In [ ]:
def chunk_text(text: str, chunk_size: int = 120, overlap: int = 20):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap

    return chunks


example_chunks = chunk_text(documents[0]["text"], chunk_size=50, overlap=20)
for number, chunk in enumerate(example_chunks, start=1):
    print(f"Chunk {number}: {chunk}\n")

# Student check: Do these chunks preserve complete ideas? Where could a boundary
# make retrieval or answer generation harder?

## Section 4: Create Embeddings

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")

## Section 5: Build a FAISS Index

In [ ]:
def build_index(chunk_size=120, overlap=20):
    chunk_texts = []
    chunk_metadata = []

    for document in documents:
        document_chunks = chunk_text(document["text"], chunk_size, overlap)
        for chunk_number, chunk in enumerate(document_chunks):
            chunk_texts.append(chunk)
            chunk_metadata.append(
                {
                    "source_id": document["id"],
                    "title": document["title"],
                    "status": document["status"],
                    "chunk_number": chunk_number,
                }
            )

    chunk_embeddings = embedding_model.encode(
        chunk_texts,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    chunk_embeddings = np.asarray(chunk_embeddings, dtype="float32")

    dimension = chunk_embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(chunk_embeddings)
    return index, chunk_texts, chunk_metadata, chunk_embeddings


index, chunk_texts, chunk_metadata, chunk_embeddings = build_index()
print("Number of chunks:", len(chunk_texts))
print("Embedding dimension:", chunk_embeddings.shape[1])
print("FAISS index size:", index.ntotal)

# Normalized vectors make inner product equivalent to cosine similarity.

## Section 6: Retrieve Relevant Chunks

In [ ]:
def retrieve(query: str, k: int = 3, active_index=None,
             active_texts=None, active_metadata=None):
    active_index = active_index or index
    active_texts = active_texts or chunk_texts
    active_metadata = active_metadata or chunk_metadata
    k = min(k, len(active_texts))

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    )
    scores, indices = active_index.search(
        np.asarray(query_embedding, dtype="float32"),
        k,
    )

    results = []
    for score, chunk_index in zip(scores[0], indices[0]):
        results.append(
            {
                "score": float(score),
                "text": active_texts[chunk_index],
                "metadata": active_metadata[chunk_index],
            }
        )
    return results


def display_retrieval(query, k=3):
    print("Question:", query)
    rows = []
    for rank, item in enumerate(retrieve(query, k), start=1):
        rows.append(
            {
                "rank": rank,
                "score": round(item["score"], 3),
                "source": item["metadata"]["title"],
                "status": item["metadata"]["status"],
                "chunk": item["text"],
            }
        )
    return pd.DataFrame(rows)


display_retrieval("How late may an assignment be submitted?", k=3)

## Section 7: Inspect Retrieval Results

For each question, inspect the rank, similarity score, source title, source status, and chunk text.

Ask:

1. Is the correct evidence in the top three?
2. Is any chunk irrelevant?
3. Did the archived policy appear?
4. Would changing k help or add noise?


In [ ]:
questions = [
    "How late may an assignment be submitted?",
    "Are exams open book?",
    "What should a student do after missing an exam because of illness?",
    "What is the required textbook price?",
]

for question in questions:
    print("\n" + "=" * 90)
    display(display_retrieval(question, k=3))

## Section 8: Construct a Grounded RAG Prompt

In [ ]:
def build_rag_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    context = "\n\n".join(
        f"[Source {i + 1}: {item['metadata']['title']} | "
        f"Status: {item['metadata']['status']}]\n{item['text']}"
        for i, item in enumerate(retrieved_chunks)
    )

    return f"""
You are a document question-answering assistant.

Answer the question using only the context below.

If the context does not support an answer, state exactly:
"The available documents do not provide enough information."

Prefer current sources over archived sources. Include the title of the source
supporting your answer.

Context:
{context}

Question:
{question}
"""


sample_question = "How late may an assignment be submitted?"
print(build_rag_prompt(sample_question, retrieve(sample_question, k=3)))

## Section 9: Generate an Answer

This notebook uses a small local instruction-following model so it does not require a paid API. The first run downloads the model and may take several minutes. You may replace this function with a university-supported API or instructor-provided output function later.

In [ ]:
USE_LOCAL_GENERATOR = True

if USE_LOCAL_GENERATOR:
    generator = pipeline(
        "text2text-generation",
        model="google/flan-t5-small",
        device=0 if torch.cuda.is_available() else -1,
    )


def generate_answer(prompt, max_new_tokens=120):
    if not USE_LOCAL_GENERATOR:
        return "Generation is disabled. Use an instructor-provided output function."
    result = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        truncation=True,
    )
    return result[0]["generated_text"]


print(generate_answer("Answer briefly: Is an exam open book?"))

## Section 10: Compare Three Conditions

In [ ]:
def build_base_prompt(question):
    return (
        "Answer this university-policy question as accurately as possible. "
        "If you are uncertain, say so.\n\nQuestion: " + question
    )


def compare_conditions(question):
    records = []
    conditions = [
        ("Base model without retrieval", build_base_prompt(question)),
        ("RAG with top-1 retrieval", build_rag_prompt(question, retrieve(question, k=1))),
        ("RAG with top-3 retrieval", build_rag_prompt(question, retrieve(question, k=3))),
    ]
    for condition, prompt in conditions:
        records.append(
            {
                "question": question,
                "condition": condition,
                "answer": generate_answer(prompt),
            }
        )
    return pd.DataFrame(records)


comparison_question = "How late may an assignment be submitted?"
condition_results = compare_conditions(comparison_question)
condition_results

### Manual comparison

For each condition, score the answer from 1 to 5 for:

- **Correctness:** Is the answer factually correct?
- **Faithfulness:** Is every claim supported by retrieved evidence?
- **Completeness:** Does it answer all parts of the question?
- **Citation quality:** Does it identify an appropriate source?


## Section 11: Retrieval Evaluation

In [ ]:
ground_truth = {
    "How late may an assignment be submitted?": {"policy_1"},
    "Are exams open book?": {"policy_2"},
    "What should a student do after missing an exam because of illness?":
        {"policy_2", "policy_3"},
    "What is the required textbook price?": set(),
}


def retrieval_metrics(query, relevant_sources, k):
    retrieved_sources = [
        item["metadata"]["source_id"] for item in retrieve(query, k=k)
    ]
    retrieved_unique = set(retrieved_sources)

    if not relevant_sources:
        return {
            "hit_at_k": None,
            "precision_at_k": None,
            "retrieved_sources": retrieved_sources,
        }

    hit_at_k = int(bool(retrieved_unique & relevant_sources))
    precision_at_k = len(retrieved_unique & relevant_sources) / k
    return {
        "hit_at_k": hit_at_k,
        "precision_at_k": precision_at_k,
        "retrieved_sources": retrieved_sources,
    }


def evaluate_retrieval(questions_to_evaluate, k):
    rows = []
    for question, relevant_sources in questions_to_evaluate.items():
        if not relevant_sources:
            continue
        metrics = retrieval_metrics(question, relevant_sources, k)
        rows.append(
            {
                "question": question,
                "Hit@k": metrics["hit_at_k"],
                "Precision@k": metrics["precision_at_k"],
                "retrieved_sources": metrics["retrieved_sources"],
            }
        )
    results = pd.DataFrame(rows)
    print(f"Mean Hit@{k}: {results['Hit@k'].mean():.2f}")
    print(f"Mean Precision@{k}: {results['Precision@k'].mean():.2f}")
    return results


retrieval_at_1 = evaluate_retrieval(ground_truth, k=1)
retrieval_at_3 = evaluate_retrieval(ground_truth, k=3)
retrieval_at_3

## Section 12: Answer Evaluation Rubric

In [ ]:
evaluation_columns = [
    "question",
    "condition",
    "correctness",
    "faithfulness",
    "relevance",
    "citation_accuracy",
    "notes",
]

evaluation_template = pd.DataFrame(
    [
        [comparison_question, condition, "", "", "", "", ""]
        for condition in condition_results["condition"]
    ],
    columns=evaluation_columns,
)

display(evaluation_template)

# After manually entering scores from 1 to 5, save your table:
# evaluation_template.to_csv("rag_answer_evaluation.csv", index=False)

## Section 13: No-Answer Questions

In [ ]:
no_answer_question = "What is the required textbook price?"
no_answer_results = compare_conditions(no_answer_question)
no_answer_results

# Evaluate whether each condition:
# - invents a price,
# - states uncertainty appropriately, or
# - cites irrelevant evidence.

## Section 14: Chunk-Size Experiment

In [ ]:
chunk_sizes = [50, 120, 250]
chunk_experiment_rows = []

for size in chunk_sizes:
    experiment_index, experiment_texts, experiment_metadata, _ = build_index(
        chunk_size=size,
        overlap=min(20, size // 2),
    )
    question = "What should a student do after missing an exam because of illness?"
    results = retrieve(
        question,
        k=3,
        active_index=experiment_index,
        active_texts=experiment_texts,
        active_metadata=experiment_metadata,
    )
    retrieved_sources = {item["metadata"]["source_id"] for item in results}
    expected_sources = ground_truth[question]
    chunk_experiment_rows.append(
        {
            "chunk_size": size,
            "number_of_chunks": len(experiment_texts),
            "Hit@3": int(bool(retrieved_sources & expected_sources)),
            "retrieved_titles": [item["metadata"]["title"] for item in results],
        }
    )

chunk_experiment = pd.DataFrame(chunk_experiment_rows)
chunk_experiment

# Discuss retrieval quality, context relevance, and likely answer quality.
# Which chunk size best balances enough evidence with low context noise?

## Section 15: Top-k Experiment

In [ ]:
top_k_rows = []
top_k_question = "What should a student do after missing an exam because of illness?"

for k in [1, 3, 5]:
    retrieved = retrieve(top_k_question, k=k)
    top_k_rows.append(
        {
            "k": k,
            "titles": [item["metadata"]["title"] for item in retrieved],
            "scores": [round(item["score"], 3) for item in retrieved],
            "context_word_count": sum(len(item["text"].split()) for item in retrieved),
        }
    )

pd.DataFrame(top_k_rows)

# Discuss evidence coverage, context noise, prompt length, and answer consistency.

## Section 16: Failure Analysis

In [ ]:
failure_analysis_template = pd.DataFrame(
    [
        ["Question", "", "", "", "", "", "", ""],
    ],
    columns=[
        "question",
        "observed_failure",
        "data_problem",
        "chunking_problem",
        "embedding_or_retrieval_problem",
        "prompt_problem",
        "generation_problem",
        "evaluation_problem",
    ],
)
failure_analysis_template

# Select one failed answer. Describe the failure and identify the most likely
# source(s) of the problem. A single answer may have more than one failure cause.

## Section 17: Reflection

Answer in a final Markdown cell:

1. Did RAG consistently outperform the base model? Why or why not?
2. Which retrieval setting worked best for this document collection?
3. Which observed failure came from retrieval rather than generation?
4. Did the model abstain appropriately for the unsupported question?
5. Which metric was most useful: Hit@k, Precision@k, correctness, faithfulness, relevance, or citation accuracy?
6. What would be needed before deploying this system in production?

## Key Takeaway

RAG quality depends on the entire pipeline. A good generator cannot reliably answer from evidence that was never retrieved, and correct retrieval does not guarantee a faithful answer. Evaluate retrieval and generation separately.
